# Part 1.4 Nodes and edges

After this notebook you can build a LangGraph graph from nodes and edges, make a model
return structured output, and route with a conditional edge.

In [ ]:
from kea.config import MODEL, setup

setup("kea-claims-local")

## See it

The finished graph from `kea.claims.graph`, run on a photo of a torn suitcase.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from kea.claims.graph import builder

store = InMemoryStore()
graph = builder.compile(checkpointer=InMemorySaver(), store=store)


def claim(photo: str, passenger_id: str, flight_number: str, declared_value_usd: float) -> dict:
    return {
        "photo": photo,  # a file name under data/bag-photos, the same input Studio takes
        "passenger_id": passenger_id,
        "flight_number": flight_number,
        "declared_value_usd": declared_value_usd,
    }

In [ ]:
from IPython.display import Image, display

from kea.claims.images import PHOTO_DIR

display(Image(PHOTO_DIR / "tear-01.jpg", width=300))
result = graph.invoke(
    claim("tear-01.jpg", "pax-1001", "KA201", 400),
    {"configurable": {"thread_id": "claim-1001"}},
)
print(result["status"], result["payout_usd"])
print(result["letter"])

In [ ]:
def show_graph(g) -> None:
    try:
        display(Image(g.get_graph().draw_mermaid_png()))
    except Exception:  # drawing needs network access
        print(g.get_graph().draw_mermaid())


show_graph(graph)

## Build it

Rebuild the first two steps. **Nodes** are functions that read the shared state and
return updates. **Edges** say which node runs next. Each LLM step returns structured
output: a Pydantic model instead of free text.

In [ ]:
from typing import Literal, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

from kea.claims import prompts
from kea.claims.images import resolve_photo
from kea.claims.models import DamageAssessment, PhotoCheck

llm = init_chat_model(MODEL)


class State(TypedDict, total=False):
    photo: str
    photo_check: dict
    assessment: dict
    letter: str


def look_at(photo: str, instructions: str) -> list:
    image = {"type": "image_url", "image_url": {"url": resolve_photo(photo)}}
    return [SystemMessage(instructions), HumanMessage([image])]


def validate_photo(state: State) -> dict:
    check = llm.with_structured_output(PhotoCheck)
    result = check.invoke(look_at(state["photo"], prompts.PHOTO_CHECK))
    return {"photo_check": result.model_dump()}


def assess_damage(state: State) -> dict:
    assess = llm.with_structured_output(DamageAssessment)
    result = assess.invoke(look_at(state["photo"], prompts.ASSESS_DAMAGE))
    return {"assessment": result.model_dump()}

In [ ]:
chain = StateGraph(State)
chain.add_node("validate_photo", validate_photo)
chain.add_node("assess_damage", assess_damage)
chain.add_edge(START, "validate_photo")
chain.add_edge("validate_photo", "assess_damage")
chain.add_edge("assess_damage", END)
two_steps = chain.compile()

car = {"photo": "car-01.jpg"}
two_steps.invoke(car)["assessment"]

Nothing stops the car photo, so the model assesses a bumper dent as luggage damage. A
**conditional edge** fixes that: a plain function reads the state and names the next
node.

In [ ]:
def reject_letter(state: State) -> dict:
    return {"letter": "We could not process this claim: the photo does not show luggage."}


def route_after_photo_check(state: State) -> Literal["assess_damage", "reject_letter"]:
    return "assess_damage" if state["photo_check"]["is_luggage"] else "reject_letter"


routed = StateGraph(State)
routed.add_node("validate_photo", validate_photo)
routed.add_node("assess_damage", assess_damage)
routed.add_node("reject_letter", reject_letter)
routed.add_edge(START, "validate_photo")
routed.add_conditional_edges("validate_photo", route_after_photo_check)
routed.add_edge("assess_damage", END)
routed.add_edge("reject_letter", END)
mini = routed.compile()

print(mini.invoke(car)["letter"])
show_graph(mini)

## Under the hood

<img src="../../assets/slides/langgraph-shared-state.jpg" width="720">

- **State is the contract between nodes.** A node never calls another node. It reads
  what it needs from the state and returns only the keys it changes; LangGraph merges
  the update.
- **Structured output** (`with_structured_output(PhotoCheck)`) returns a validated
  Pydantic object, so the router can trust that `is_luggage` is a real boolean.
- **The router is ordinary Python.** Its `Literal` return type is also what draws the
  branches in the diagram.
- **`apply_policy` is plain Python on purpose.** The payout table is a rule, not a
  judgment call: the same claim must get the same answer, and a test can prove it. Use
  the model only where the input is fuzzy. See `kea/claims/policy.py`.

Docs: [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api),
[Structured output](https://docs.langchain.com/oss/python/langchain/structured-output).

## Recap

- Nodes read state and return updates; edges choose what runs next.
- Structured output makes a model's answer safe to route on.
- A conditional edge is a function that returns the next node's name.
- Keep rules in Python, not in prompts.

**Next:** [Part 1.5 Tracing](05_tracing.ipynb)